### Display clustering results
Notebook to compare two metrics for clustering: MTEB, which uses sklearn.cluster.MiniBatchKMeans() and kNN, which uses sklearn.neighbors.KNeighborsClassifier()
MTEB results are saved in .json files, one for each task. 
kNN results are saved in .json files, one for each model.
This notebook aggregates the results for multiple tasks and multiple models.

In [ ]:
import os
import mteb
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import regex as re


In [ ]:
# get task type 
mteb.get_task("ArguAna").metadata.type

In [ ]:
mteb.get_task("ArguAna").metadata.name

#### Analyze MTEB results of different TF-IDF configurations

In [ ]:
data = {}
main_dir = "../MTEB/sparse_results"

# Traverse folders and subfolders
for (root, dirs, files) in os.walk(main_dir):
    # Identify model version from the folder structure
    model_version = os.path.basename(root)
    #print(model_version)
        
    for file in files:
        # Skip unwanted files
        if file in {"model_meta.json"} or not file.endswith('.json'):
            continue
            
        file_path = os.path.join(root, file)
        try:
            # Read JSON and extract necessary fields
            with open(file_path, 'r') as f:
                json_data = json.load(f)
                task_name = json_data.get("task_name")
                main_score = json_data.get("scores", {}).get("test", {})[0]["main_score"]
                main_score = round(main_score*100, 2)
                    
                if task_name and main_score is not None:
                    if task_name not in data:
                        data[task_name] = {}
                    data[task_name][model_version] = main_score
        except (json.JSONDecodeError, KeyError):
            print(f"Error parsing file: {file_path}")


In [ ]:
df_mteb = pd.DataFrame.from_dict(data, orient='index')
df_mteb.index.name = "task_name"

In [ ]:
task_selection = ["ArxivClusteringP2P", "BiorxivClusteringP2P", "MedrxivClusteringP2P",
                 "RedditClusteringP2P", "StackExchangeClusteringP2P"]
exclude_models = ["no_revision_available", "svd", "svd_log_run1", "tfidf_log_run1", "svd_log_old_run2"]

df_mteb_c = df_mteb.loc[task_selection].drop(exclude_models, axis=1)
df_mteb_c

#### Analyze kNN results of different TF-IDF configurations

In [ ]:
data = {}
main_dir = "../MTEB/knn_results"

# Traverse folders and subfolders
for (root, dirs, files) in os.walk(main_dir):
        
    for file in files:
        # Skip unwanted files
        model_name = file.strip(".json")
            
        file_path = os.path.join(root, file)
        try:
            # Read JSON and extract necessary fields
            with open(file_path, 'r') as f:
                json_data = json.load(f)
                for key, value in json_data.items():
                    if type(value) == list:
                        value = np.mean(value)
                    json_data[key] = round(value*100, 2)

                data[model_name.removeprefix("Tfidf_")] = json_data
        except (json.JSONDecodeError, KeyError):
            print(f"Error parsing file: {file_path}")


In [ ]:
df_knn = pd.DataFrame.from_dict(data)
df_knn